In [ ]:
# Libs

from pmdarima import auto_arima
from sqlalchemy import create_engine
import pandas as pd

# Database connection

engine = create_engine(
    "postgresql+psycopg2://postgres:@localhost:5432/Online Retail II UCI"
)


### ARIMA — Overview

ARIMA (AutoRegressive Integrated Moving Average) is a classical statistical
method for time series forecasting. Unlike Prophet, it does not automatically
detect trend and seasonality,  it requires you to understand and configure
your time series through three parameters: ARIMA(p, d, q)

- p (AutoRegressive)  → how much the future depends on past values
                         p=2 means the model looks at the last 2 periods

- d (Integrated)       → how many times the series is differenced to make it
                          stationary (stable mean and variance over time)
                         d=1 means using (value[t] - value[t-1])

- q (Moving Average)   → how much the model corrects itself based on past
                         forecast errors

auto_arima (from pmdarima) automatically tests multiple combinations of
p, d, q and selects the best one, this is the entry point for learning
ARIMA before tuning parameters manually.

### Key difference from Prophet:
- Prophet expects a DataFrame with columns ds and y
- ARIMA only needs the y values as a sequence, dates are not part of
the model, they are reattached afterward for readability

### ARIMA works in two steps:
1. Train   → model = auto_arima(df['y'], seasonal=True, m=52)
2. Predict → forecast, conf_int = model.predict(n_periods=periods, return_conf_int=True)

In [ ]:
# Product revenue aggregated by week (valid_product only)

df_products_weekly_a = pd.read_sql("""
    SELECT 
        DATE_TRUNC('week', invoice_date)::date AS ds,
        SUM(revenue) AS y	
    FROM fact_transactions ft
    INNER JOIN dim_products dp ON ft.stock_code = dp.stock_code
    WHERE most_freq_sctype = 'valid_product'
    GROUP BY DATE_TRUNC('week', invoice_date)::date
    ORDER BY DATE_TRUNC('week', invoice_date)::date ASC
""", engine)

In [ ]:
# Reusable ARIMA pipeline
# m=52 is fixed (yearly seasonality on weekly data), independent from `periods`
# which controls how many future weeks to forecast
# Future dates use freq='W-MON' to match PostgreSQL's DATE_TRUNC('week', ...),
# which truncates to Monday, pandas defaults to Sunday otherwise

def run_arima(df, periods):    
    model = auto_arima(
        df['y'],
        seasonal=True, # detects seasonality (ARIMA -> S(easonal)ARIMA)
        m=52, # seasonality frequence
        suppress_warnings=True # ignores warnings during training
    )

    yhat, conf_int = model.predict(
        n_periods=periods,
        return_conf_int=True
    )


    # Generate future dates only, ARIMA does not return historical predictions,
    # so the resulting df_fc contains forecast rows exclusively (no past data, no y)

    dates = pd.date_range(start=df['ds'].max() + pd.to_timedelta(7, unit='d'), periods=periods, freq='W-MON') # weeks that ends/starts monday to match the grouping of PostgreSQL
    df_fc = pd.DataFrame({
        "ds":dates,
        "yhat":yhat,
        "yhat_lower":conf_int[:,0],
        "yhat_upper":conf_int[:,1],
    })

    df_fc['model'] = 'ARIMA'
    df_fc['granularity'] = 'weekly'

    return df_fc

In [ ]:
# ## Final model — Products
# Trained on 100% of available data, matching the same production approach used for Prophet. Forecast horizon: 52 weeks (1 year)

df_products_weekly_fca = run_arima(df_products_weekly_a,52)

In [ ]:
df_products_weekly_fca.to_sql(
    name='forecast_results_products_arima',
    con=engine,
    if_exists='append',
    index=False
)

In [ ]:
df_business_weekly_a = pd.read_sql("""
SELECT 
    DATE_TRUNC('week', invoice_date)::date AS ds,
    SUM(revenue) AS y	
FROM fact_transactions ft
INNER JOIN dim_products dp ON ft.stock_code = dp.stock_code
WHERE most_freq_sctype LIKE 'valid%%'
GROUP BY DATE_TRUNC('week', invoice_date)::date
ORDER BY DATE_TRUNC('week', invoice_date)::date ASC
""", engine)

In [ ]:
# ## Final model — Business
# Trained on 100% of available data, matching the same production approach used for Prophet. Forecast horizon: 52 weeks (1 year)

df_business_weekly_fca = run_arima(df_business_weekly_a, 52)

In [ ]:
df_business_weekly_fca.to_sql(
    name='forecast_results_business_arima',
    con=engine,
    if_exists='append',
    index=False
)

### Validating the ARIMA methodology — Train/Test Split

Unlike Prophet, ARIMA never reconstructs historical predictions, so its
accuracy cannot be measured directly against the training data. 

A holdout set is required: train on an earlier portion of the data, predict forward, and compare against the actual values that were deliberately held back.

Split: 72 weeks train (~1.4 years, just over one full seasonal cycle)
       32 weeks test  (~6 months)

In [ ]:
df_train = df_products_weekly_a.iloc[0:72]
df_test = df_products_weekly_a.iloc[72:]

In [ ]:
periods = df_test.shape[0]
df_products_weekly_test_fca = run_arima(df_train,periods)

In [ ]:
df_test['ds'] = pd.to_datetime(df_test['ds'])

In [ ]:
df_products_test_final = df_products_weekly_test_fca.merge(df_test, how='inner', left_on='ds', right_on='ds')

In [ ]:
def calculate_mae(df):
    mae = abs(df['y'] - df['yhat']).mean()
    mean = df['y'].mean()
    return mae, mean

In [ ]:
mae_products_test, mean_products_test = calculate_mae(df_products_test_final)
print(f'Products test MAE is {round(mae_products_test,2)} & MEAN is {round(mean_products_test,2)}')

### Summary

ARIMA forecasting on weekly product and business revenue, using auto_arima
to select (p, d, q) and detect yearly seasonality (m=52).

- run_arima(df, periods) trains the model and returns a forecast DataFrame
(ds, yhat, yhat_lower, yhat_upper) aligned with PostgreSQL's weekconvention (freq='W-MON') and Prophet's column naming.

- Final models (products, business) trained on 100% of the data and
exported to forecast_results_products_arima / _business_arima.

- Validated separately via a 72/32 week train/test split, since ARIMA only
forecasts forward and never reconstructs the past. Result: MAE ≈59,108
on a mean of ≈205,273 (≈29% error), higher than Prophet's ≈15%, likely
because 72 weeks barely covers one full seasonal cycle (52 weeks).

### Decision: 
The split validated the methodology but isn't suited for the
final model given limited data. Final models use 100% of the data, no
holdout, same approach as Prophet. Both are exported separately, tagged
by model and granularity, for comparison in Power BI.